## OLLAMA 

### Python SDK

In [20]:
import ollama

output = ollama.chat(
    model = 'qwen3.5:2b',
    messages=[
        {"role": "user", 
        "content": "What is the capital of Scotland? and has it ever changed?"
        }
    ]
)

### Thinking

In [28]:
print(output['message']['thinking'])

Thinking Process:

1.  **Analyze the Request:**
    *   Question 1: What is the capital of Scotland?
    *   Question 2: Has it ever changed?

2.  **Fact-Check Fact 1: Capital of Scotland.**
    *   Knowledge retrieval: The capital of Scotland is Glasgow (not Edinburgh - wait, that's a common mix-up). Wait, Edinburgh is the capital of England and Wales? No, that's wrong.
    *   Correction: Let me recall. The capital of Scotland (the UK nation) is actually often confused with London. In Scotland, the main city historically and culturally... wait, no.
    *   Current Knowledge: The capital of the United Kingdom is Westminster/London. Scotland has its own capital? No, Scotland has an administration capital. But historically, the capital of Great Britain was Edinburgh. Now, who is the capital of Scotland?
    *   Let's check: Is it Edinburgh or Glasgow? Or Stirling?
    *   Correction: **Edinburgh** is the historic capital. However, **Glasgow** is the most populous city. Wait. Which is th

### LLM Backend

Creating an interface orthogonal to the LLM

In [2]:
class LLMBackend:
    def generate(self, messages):
        raise NotImplementedError

In [ ]:
import ollama


class OllamaBackend(LLMBackend):
    def __init__(self, model):
        self.model = model

    def generate(self, messages: list[dict[str, str]]) -> str:
        output = ollama.chat(
            model=self.model,
            messages=messages,
            think=False,
            stream=False,
        )
        
        content = output['message']['content']

        if not content:
            raise ValueError("No content returned from the model.")
        return content            




In [39]:
llm = OllamaBackend(model="qwen3.5:2b")
output= llm.generate(
    messages=[
        {"role": "user", 
        "content": "What is the capital of Iran? and has it ever changed?"}
        ]
    )

In [40]:
print(output)

The capital of Iran is **Tehran**.

**Has it ever changed?**
Yes, Tehran's status as the capital has evolved several times:

1.  **Before the Revolution**: Before the 1979 Islamic Revolution, Tehran was actually the *second* capital of Iran (after Kashan). The first capital was Shiraz. Following the revolution in 1979, Tehran became the undisputed capital.
2.  **During the Pahlavi Period (Pre-Revolution)**: From about 1951 to 1964, while Mohammad Reza Pahlavi (the Shah) ruled, Tehran served as both the capital and the official residence of the monarch. After he was ousted in 1979, it reverted to being only the second capital.
3.  **Modern Era**: Currently, since the restoration of the monarchy in 2005 and the eventual establishment of a republic in 1979, Tehran has remained the single, official capital city without interruption.


### Minimal Agentic Conversation Loop

In [ ]:
MODEL_NAME = "qwen3.5:2b"
llm = OllamaBackend(model=MODEL_NAME)
messages = []

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        break

    messages.append({"role": "user", "content": user_input})
    output = llm.generate(messages=messages)

    messages.append({"role": "assistant", "content": output})
    print(f"Assistant: {output}")

## Model Benchmarking

Later we can use these metrics to compare different LLMs

In [ ]:
from dataclasses import dataclass

import ollama


@dataclass
class LLMResponse:
    content: str
    total_time_s: float
    load_time_s: float
    generated_tokens: int
    tokens_per_second: float


class LLMBackend:
    def generate(self, messages):
        raise NotImplementedError
    

class OllamaBackend(LLMBackend):
    def __init__(self, model):
        self.model = model

    def generate(self, messages: list[dict]) -> LLMResponse:
        output = ollama.chat(
            model=self.model,
            messages=messages,
            think=False,
            stream=False,
            options={"num_ctx": 8192, "num_predict": 800}
        )
        
        content = output['message']['content']
        total_time_s = output['total_duration'] / 1e9
        load_time_s = output['load_duration'] / 1e9
        generated_tokens = output.eval_count
        generation_time = output.eval_duration / 1e9

        tokens_per_second = (
            generated_tokens / generation_time
            if generation_time > 0 else 0
        )

        if not content:
            raise ValueError("No content returned from the model.")
        return LLMResponse(
            content=content,
            total_time_s=total_time_s,
            load_time_s=load_time_s,
            generated_tokens=generated_tokens,
            tokens_per_second=tokens_per_second
        )



In [ ]:
MODEL_NAME = "gemma4:e2b"  # "qwen3.5:2b"
llm = OllamaBackend(model=MODEL_NAME)
messages = []
MAX_HISTORY_MESSAGES = 8


while True:
    user_input = input("You: ").strip()
    if user_input.lower() in {"exit", "quit"}:
        break

    messages.append({"role": "user", "content": user_input})

    remaining_messages = messages[-MAX_HISTORY_MESSAGES:]
    output = llm.generate(messages=remaining_messages)

    messages.append({"role": "assistant", "content": output.content})
    print(f"Assistant: {output.content}")
    print(f"Total time (s): {output.total_time_s}")
    print(f"Load time (s): {output.load_time_s}")
    print(f"Generated tokens: {output.generated_tokens}")
    print(f"Tokens per second: {output.tokens_per_second}")
    